In [1]:
"""
A financial institution wants to predict whether a customer will default on a loan before approving it. Early identification of
risky customers helps reduce financial loss. You are working as a Machine Learning Analyst and must build a classification model
using the K-Nearest Neighbors (KNN) algorithm to predict loan default.

This case introduces:
Mixed feature types
Financial risk interpretation
Class imbalance awareness

Age,AnnualIncome(lakhs),CreditScore(300-900), LoanAmount(lakhs), LoanTerm(years), EmploymentType, loan(yes/no)
28,6.5,720,5,5,Salaried,0
45,12,680,10,10,Self-Employed,1
35,8,750,6,7,Salaried,0
50,15,640,12,15,Self-Employed,1
30,7,710,5,5,Salaried,0
42,10,660,9,10,Salaried,1
26,5.5,730,4,4,Salaried,0
48,14,650,11,12,Self-Employed,1
38,9,700,7,8,Salaried,0
55,16,620,13,15,Self-Employed,1

Interpretation
Identify high-risk customers.
What patterns lead to loan default?
How do credit score and income influence predictions?
Suggest banking policies based on model output.
Compare KNN with Decision Trees for this problem.
What happens if LoanAmount dominates distance calculation?
Should KNN be used in real-time loan approval systems?
"""

'\nA financial institution wants to predict whether a customer will default on a loan before approving it. Early identification of\nrisky customers helps reduce financial loss. You are working as a Machine Learning Analyst and must build a classification model\nusing the K-Nearest Neighbors (KNN) algorithm to predict loan default.\n\nThis case introduces:\nMixed feature types\nFinancial risk interpretation\nClass imbalance awareness\n\nAge,AnnualIncome(lakhs),CreditScore(300-900), LoanAmount(lakhs), LoanTerm(years), EmploymentType, loan(yes/no)\n28,6.5,720,5,5,Salaried,0\n45,12,680,10,10,Self-Employed,1\n35,8,750,6,7,Salaried,0\n50,15,640,12,15,Self-Employed,1\n30,7,710,5,5,Salaried,0\n42,10,660,9,10,Salaried,1\n26,5.5,730,4,4,Salaried,0\n48,14,650,11,12,Self-Employed,1\n38,9,700,7,8,Salaried,0\n55,16,620,13,15,Self-Employed,1\n\nInterpretation\nIdentify high-risk customers.\nWhat patterns lead to loan default?\nHow do credit score and income influence predictions?\nSuggest banking pol

In [2]:
import pandas as pd

dataset_file = 'ps-2.csv'

# Prepare the dataset
try:
    df = pd.read_csv(dataset_file)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("File not found!")

df.head()

Dataset loaded successfully.


,Age,AnnualIncome(lakhs),CreditScore(300-900),LoanAmount(lakhs),LoanTerm(years),EmploymentType,loan(yes/no)
0,28,6.5,720,5,5,Salaried,0
1,45,12.0,680,10,10,Self-Employed,1
2,35,8.0,750,6,7,Salaried,0
3,50,15.0,640,12,15,Self-Employed,1
4,30,7.0,710,5,5,Salaried,0


In [8]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Encode Categorical Data (Employment)
label_encoder = LabelEncoder()
df['EmploymentType'] = label_encoder.fit_transform(df['EmploymentType']) # Salaried=0, Self-Employed=1

# Features and Target. Assumed 'loan(yes/no)' column as 'defaulting on the loan'/'successful loan'.
X = df.drop('loan(yes/no)', axis=1)
y = df['loan(yes/no)']

## Not splitting the data into train-test sets as the number of samples are small

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Build KNN Model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_scaled, y)

# Results
df['Risk_Prob'] = knn.predict_proba(X_scaled)[:, 1]
df['loan-vs-income'] = df['LoanAmount(lakhs)'] / df['AnnualIncome(lakhs)']
print("\n--- Loan Risk Analysis (KNN) ---")
df.sort_values(by='Risk_Prob', ascending=False).head(10)


--- Loan Risk Analysis (KNN) ---


,Age,AnnualIncome(lakhs),CreditScore(300-900),LoanAmount(lakhs),LoanTerm(years),EmploymentType,loan(yes/no),Risk_Prob,loan-vs-income
1,45,12.0,680,10,10,1,1,1.000000,0.833333
3,50,15.0,640,12,15,1,1,1.000000,0.800000
9,55,16.0,620,13,15,1,1,1.000000,0.812500
7,48,14.0,650,11,12,1,1,1.000000,0.785714
5,42,10.0,660,9,10,0,1,0.666667,0.900000
8,38,9.0,700,7,8,0,0,0.333333,0.777778
0,28,6.5,720,5,5,0,0,0.000000,0.769231
2,35,8.0,750,6,7,0,0,0.000000,0.750000
4,30,7.0,710,5,5,0,0,0.000000,0.714286
6,26,5.5,730,4,4,0,0,0.000000,0.727273


In [9]:
"""
Identify high-risk customers.

High-risk customers in this dataset are characterized by 'Self-Employment' as EmploymentType, Credit Scores below 680, Age greater
than 40 and longer loan terms (>10 years). Even with higher incomes, these individuals defaulted in the dataset.
"""

"\nIdentify high-risk customers.\n\nHigh-risk customers in this dataset are characterized by 'Self-Employment' as EmploymentType, Credit Scores below 680, Age greater\nthan 40 and longer loan terms (>10 years). Even with higher incomes, these individuals defaulted in the dataset.\n"

In [10]:
"""
What patterns lead to loan default?

A High Loan to Income ratio combined with a declining Credit Score. Also as Age and Income increase, the Loan Amount also
increases significantly, but the Credit Score drops.
"""

'\nWhat patterns lead to loan default?\n\nA High Loan to Income ratio combined with a declining Credit Score. Also as Age and Income increase, the Loan Amount also\nincreases significantly, but the Credit Score drops.\n'

In [11]:
"""
How do credit score and income influence predictions?

Here, the Credit Score is the stronger predictor of defaulting. Even a customer with 16 Lakhs Annual Income (highest) defaulted
because their Credit Score was below 700.
"""

'\nHow do credit score and income influence predictions?\n \nHere, the Credit Score is the stronger predictor of defaulting. Even a customer with 16 Lakhs Annual Income (highest) defaulted \nbecause their Credit Score was below 700.\n'

In [13]:
"""
Suggest banking policies based on model output.

Policy 1: Implement a Hard Floor for Credit Scores (e.g., no approvals below 700).
Policy 2: Cap the loan term times (100% defaults for terms greater than equal to 10 years).
"""

'\nSuggest banking policies based on model output.\n\nPolicy 1: Implement a Hard Floor for Credit Scores (e.g., no approvals below 700).\nPolicy 2: Cap the loan term times (100% defaults for terms greater than equal to 10 years).\n'

In [14]:
"""
Compare KNN with Decision Trees for this problem.

KNN: It doesn't learn rules, it looks at similar past cases. It is computationally expensive for prediction
(order of n*d algorithm, where n=datapoints and d=dimensions).

Decision Trees: Create logic gates (e.g., If Credit less than 700, then default). They handle mixed data types
(Salaried vs. Self-Employed) much better than KNN and are computationally less expensive for prediction.
"""

"\nCompare KNN with Decision Trees for this problem.\n\nKNN: It doesn't learn rules, it looks at similar past cases. It is computationally expensive for prediction\n(order of n*d algorithm, where n=datapoints and d=dimensions).\n\nDecision Trees: Create logic gates (e.g., If Credit less than 700, then default). They handle mixed data types \n(Salaried vs. Self-Employed) much better than KNN and are computationally less expensive for prediction.\n"

In [15]:
"""
What happens if LoanAmount dominates distance calculation?

If the data wasn't scaled, the model will think a difference of 1 Lakh in LoanAmount is the same as a 1 point difference in
CreditScore. Since CreditScore ranges from 300–900 and LoanAmount from 4–13, the CreditScore would actually dominate. Scaling
ensures that a large change in LoanAmount is treated with the same importance as the change in Credit Score.
"""

"\nWhat happens if LoanAmount dominates distance calculation?\n\nIf the data wasn't scaled, the model will think a difference of 1 Lakh in LoanAmount is the same as a 1 point difference in \nCreditScore. Since CreditScore ranges from 300–900 and LoanAmount from 4–13, the CreditScore would actually dominate. Scaling \nensures that a large change in LoanAmount is treated with the same importance as the change in Credit Score.\n"

In [16]:
"""
Should KNN be used in real-time loan approval systems?

KNN is computationally expensive (order of n*d algorithm, where n=datapoints and d=dimensions) for prediction. If there are
1 million customers, KNN has to calculate the distance to all 1 million people every time someone applies for a loan.
For real-time systems, Logistic Regression or XGBoost (Decision Tree type algorithm) should be preferred for speed.
"""

'\nShould KNN be used in real-time loan approval systems?\n\nKNN is computationally expensive (order of n*d algorithm, where n=datapoints and d=dimensions) for prediction. If there are \n1 million customers, KNN has to calculate the distance to all 1 million people every time someone applies for a loan. \nFor real-time systems, Logistic Regression or XGBoost (Decision Tree type algorithm) should be preferred for speed.\n'